In [1]:
!pip install pandas

In [14]:
import pandas as pd
import numpy as np
import re

In [4]:
df= pd.read_csv("news_category_dataset.xls")

In [ ]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [10]:
df['category'].value_counts()

category
POLITICS          35600
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6346
FOOD & DRINK       6340
BUSINESS           5991
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3571
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2943
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2578
RELIGION           2576
STYLE              2254
SCIENCE            2206
TECH               2100
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1443
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
# Drop rows with missing headline or category
df.dropna(subset=['headline', 'category'], inplace=True)

In [8]:
# Normalize text columns
text_cols = ['headline', 'category', 'short_description', 'authors']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.encode('utf-8', 'ignore').str.decode('utf-8')

In [ ]:
# Filter for 10 selected categories
selected_categories = [
    'POLITICS', 'WELLNESS', 'ENTERTAINMENT', 'TRAVEL', 'STYLE & BEAUTY',
    'PARENTING', 'FOOD & DRINK', 'BUSINESS', 'COMEDY', 'WORLD NEWS'
]
df = df[df['category'].isin(selected_categories)]

In [12]:
# Convert date to datetime ---
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [ ]:
# Text preprocessing for NLP tasks
def clean_text(text):
    text = text.lower()                               # Lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)           # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()          # Remove extra spaces
    return text

In [16]:
df['headline_clean'] = df['headline'].apply(clean_text)
df['short_description_clean'] = df['short_description'].apply(clean_text)

In [17]:
df.to_csv("cleaned_dataset.csv", index=False)

In [18]:
print("Cleaned dataset shape:", df.shape)
print(df['category'].value_counts())
print(df.head())

Cleaned dataset shape: (120436, 8)
category
POLITICS          35600
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
PARENTING          8791
FOOD & DRINK       6340
BUSINESS           5991
COMEDY             5400
WORLD NEWS         3299
Name: count, dtype: int64
                                                 link  \
2   https://www.huffpost.com/entry/funniest-tweets...   
3   https://www.huffpost.com/entry/funniest-parent...   
7   https://www.huffpost.com/entry/puerto-rico-wat...   
9   https://www.huffpost.com/entry/biden-un-russia...   
10  https://www.huffpost.com/entry/bc-soc-wcup-cap...   

                                             headline    category  \
2   23 Of The Funniest Tweets About Cats And Dogs ...      COMEDY   
3   The Funniest Tweets From Parents This Week (Se...   PARENTING   
7   Puerto Ricans Desperate For Water After Hurric...  WORLD NEWS   
9   Biden At UN To Call Russian War An Affront To ...  WORLD NEWS   
10

In [19]:
df.head()

,link,headline,category,short_description,authors,date,headline_clean,short_description_clean
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23,23 of the funniest tweets about cats and dogs ...,until you have a dog you dont understand what ...
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23,the funniest tweets from parents this week sep...,accidentally put grownup toothpaste on my todd...
7,https://www.huffpost.com/entry/puerto-rico-wat...,Puerto Ricans Desperate For Water After Hurric...,WORLD NEWS,More than half a million people remained witho...,"DÁNICA COTO, AP",2022-09-22,puerto ricans desperate for water after hurric...,more than half a million people remained witho...
9,https://www.huffpost.com/entry/biden-un-russia...,Biden At UN To Call Russian War An Affront To ...,WORLD NEWS,White House officials say the crux of the pres...,"Aamer Madhani, AP",2022-09-21,biden at un to call russian war an affront to ...,white house officials say the crux of the pres...
10,https://www.huffpost.com/entry/bc-soc-wcup-cap...,World Cup Captains Want To Wear Rainbow Armban...,WORLD NEWS,FIFA has come under pressure from several Euro...,"GRAHAM DUNBAR, AP",2022-09-21,world cup captains want to wear rainbow armban...,fifa has come under pressure from several euro...


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

In [21]:
df= pd.read_csv("cleaned_dataset.csv")

In [22]:
df.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date',
       'headline_clean', 'short_description_clean'],
      dtype='object')

In [23]:
X = df['headline_clean'] if 'headline_clean' in df.columns else df['headline']
y = df['category']

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [25]:
# Convert text to TF-IDF features
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [26]:
model= MultinomialNB()
model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [27]:
y_pred= model.predict(X_test_tfidf)

In [28]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7267934241115909
Classification Report:
                 precision    recall  f1-score   support

      BUSINESS       0.75      0.28      0.41      1198
        COMEDY       0.77      0.29      0.42      1080
 ENTERTAINMENT       0.71      0.76      0.73      3473
  FOOD & DRINK       0.78      0.66      0.72      1268
     PARENTING       0.79      0.56      0.65      1758
      POLITICS       0.71      0.94      0.81      7120
STYLE & BEAUTY       0.83      0.75      0.79      1962
        TRAVEL       0.77      0.70      0.73      1980
      WELLNESS       0.67      0.76      0.71      3589
    WORLD NEWS       0.81      0.22      0.35       660

      accuracy                           0.73     24088
     macro avg       0.76      0.59      0.63     24088
  weighted avg       0.74      0.73      0.71     24088



In [29]:
# Example Prediction
sample = ["Biden meets world leaders at UN summit"]
sample_tfidf = tfidf.transform(sample)
predicted_category = model.predict(sample_tfidf)[0]
print(f"Predicted Category for sample: {predicted_category}")

Predicted Category for sample: POLITICS
